# Detect domain-specific hallucinations

When a support bot promises the wrong return window, did it retrieve the wrong policy or invent an exception? Use synthetic Ridgeline Outfitters policies to separate those two failure modes, enforce a domain rule, and check a revised answer. You need Future AGI credentials; only the final revision uses OpenAI.

## Install and authenticate

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7", "openai"], check=True);

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    value = getpass(f"{name}: ")
    if not value.strip():
        raise ValueError(f"{name} is required")
    os.environ[name] = value.strip()

for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    load_secret(name)

## Build a small regression set

Treat the text actually retrieved for each question as the source of truth for the groundedness judge. Keep the intended policy and human label separately: the wrong document can still support a factually accurate answer to the wrong question.

In [ ]:
policies = {
    "returns": "Unused Ridgeline jackets can be returned within 30 days of delivery. The original receipt is required. There is no holiday extension.",
    "repairs": "Ridgeline offers zipper repairs on Summit packs for 60 days after delivery. Bring the pack to a repair counter.",
}
question = "How long do I have to return my unused Ridgeline jacket?"
cases = [
    {"label": "supported", "source": "returns", "answer": "You can return your unused jacket within 30 days of delivery."},
    {"label": "invented window", "source": "returns", "answer": "You have 90 days after delivery to return an unused jacket."},
    {"label": "wrong document", "source": "repairs", "answer": "Ridgeline offers zipper repairs on Summit packs for 60 days after delivery."},
]

## Score retrieval separately from generation

`context_relevance` judges the retrieved text against the question. `groundedness` judges whether that text supports the answer. Keep the hosted result IDs to look up the original verdicts without rerunning the judges. If the service returns HTTP 429, retry only that evaluation after a wait, up to twice. A failed request is not a verdict.

In [ ]:
from time import sleep
from fi.evals import Evaluator, evaluate

hosted_ids = []

def score(metric, **inputs):
    for delay in (0, 30, 60):
        if delay:
            sleep(delay)
        result = evaluate(metric, model="turing_small", **inputs)
        if result.passed is not None and not result.error:
            hosted_ids.append((result.metadata["eval_id"], metric))
            return result
        message = result.error or result.reason or "no verdict returned"
        if not message.startswith("Error in evaluation: 429,"):
            break
    raise RuntimeError(f"No {metric} verdict: {message}")

for case in cases:
    source = policies[case["source"]]
    relevance = score("context_relevance", input=question, context=source)
    grounded = score("groundedness", input=question, context=source, output=case["answer"])
    print(case["label"], "| relevant:", relevance.passed, "| grounded:", grounded.passed)
    print("  Retrieval:", relevance.reason)
    print("  Answer:", grounded.reason)

## Enforce the return-window rule

For this particular question, the support policy requires a 30-day answer. A numeric rule catches a changed window even when a judge misses it. It checks only the number and unit, not whether the rest of the answer is correct.

In [ ]:
import re

def states_only_30_days(text):
    windows = re.findall(r"\b(\d+)\s*[- ]?\s*days?\b", text, flags=re.IGNORECASE)
    return bool(windows) and all(n == "30" for n in windows)

for case in cases:
    print(case["label"], "| return-window rule:", states_only_30_days(case["answer"]))
assert [states_only_30_days(c["answer"]) for c in cases] == [True, False, False]

## Repair the prompt and recheck the answer

Use the intended return policy, ask the model not to supply missing details, and run the same checks on what it actually says. A new model response is not guaranteed to pass.

In [ ]:
from openai import OpenAI

reply = OpenAI().chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer only from the supplied policy. Keep numbers exact. If the policy does not answer the question, say so."},
        {"role": "user", "content": f"Policy: {policies['returns']}\nQuestion: {question}"},
    ],
)
answer = reply.choices[0].message.content
if not answer:
    raise ValueError("No text answer")
verdict = score("groundedness", input=question, context=policies["returns"], output=answer)
print(answer)
print("Supported:", verdict.passed, "| return-window rule:", states_only_30_days(answer))

## Retrieve the saved verdicts

In [ ]:
history = Evaluator()
for eval_id, metric in hosted_ids:
    record = history.get_eval_result(eval_id)["result"]["result"]
    assert record["eval_id"] == eval_id and record["name"] == metric
    print(metric, eval_id, record["value"])

Inspect the retrieved document before editing the prompt. If the retrieval is wrong, no wording change can make the missing policy appear in the source.